# SoftCom myopia-risk benchmark - Section 3 - Multi-model benchmarking

Replication and extension of Mu et al. 2024 (Front. Med. 11:1482788) on the provided **hybrid cohort (N=1500)**. Pipeline validation only - no clinical claims.

**Pipeline:** SoftCom_01_Data_Validation -> SoftCom_02_Preprocessing -> **SoftCom_03_Benchmarking** -> SoftCom_04_Explainable_AI -> SoftCom_05_Figures -> SoftCom_06_Export

All inputs and outputs live in Google Drive at `MyDrive/SoftCom_Myopia/`. Each notebook checks `artifacts/manifest.json` and stops if an upstream step has not run or its files have changed.

| | |
|---|---|
| **Reads** | outputs of 02 |
| **Writes** | `models/*.joblib`, `predictions/*.csv`, `tables/table2*`, `table3_delong_pairwise.csv`, `artifacts/benchmark_meta.json`, `fold_indices.json`, `figures/training_curves_*` |
| **Next** | `SoftCom_04_Explainable_AI.ipynb` |

Runtime: Colab CPU is enough. Use *Runtime -> Run all*; accept the Drive prompt in the setup cell. For a quick trial run, set `"FAST_MODE": True` in the CONFIG cell of **every** notebook (fewer CV repeats/bootstraps); set it back to `False` for the reported results.

## 0. Install dependencies and set up Google Drive

In [ ]:
%pip install -q "shap>=0.47" xgboost lightgbm statsmodels imbalanced-learn

In [ ]:
"""Shared setup (identical in every SoftCom notebook).

Mounts Google Drive, defines CONFIG, the Drive folder layout, the run manifest
that links the six notebooks together, seeding, logging and figure helpers.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ----------------------------------------------------------------------------- CONFIG
CONFIG: Dict[str, Any] = {
    "SEED": 42,
    "TEST_SIZE": 0.3,
    "N_SPLITS": 5,
    "N_REPEATS": 10,
    "N_BOOTSTRAP": 2000,
    "EXPECTED_N_ROWS": 1500,
    "DATA_FILENAME": "full_myopia_hybrid_dataset_N1500.csv",
    "PILOT_FILENAME": "pilot_real.csv",          # optional 73-row pilot; never merged
    "TARGET": "is_myopic",
    "SOURCE_LABEL": "hybrid",
    "COHORT_LABEL": "Hybrid cohort (N=1500) - pipeline validation only, not clinical evidence",
    "DRIVE_PROJECT_DIR": "SoftCom_Myopia",       # folder inside MyDrive holding everything
    "USE_SMOTE": False,                          # SMOTE inside pipeline only (imblearn)
    "USE_LASSO_SELECTION": True,                 # L1-logistic SelectFromModel inside pipeline
    "ABLATION": True,                            # also run FT-Transformer without attention
    "DEEP_MAX_EPOCHS": 200,
    "DEEP_PATIENCE": 20,
    "DEEP_BATCH_SIZE": 64,
    "PERM_N_REPEATS": 20,
    "SHAP_BACKGROUND": 100,
    "SHAP_N_EXPLAIN": 150,
    "SHAP_KERNEL_NSAMPLES": 200,
    "FAST_MODE": False,                          # True = quick trial run (same value in all notebooks)
}
CONFIG["FAST_MODE"] |= os.environ.get("SOFTCOM_FAST", "0") == "1"  # local smoke tests
if CONFIG["FAST_MODE"]:
    CONFIG.update(N_REPEATS=2, N_BOOTSTRAP=200, DEEP_MAX_EPOCHS=25, DEEP_PATIENCE=5,
                  PERM_N_REPEATS=3, SHAP_BACKGROUND=20, SHAP_N_EXPLAIN=20,
                  SHAP_KERNEL_NSAMPLES=60)
SEED: int = CONFIG["SEED"]

NOTEBOOKS: Dict[str, str] = {
    "01_data": "SoftCom_01_Data_Validation.ipynb",
    "02_preprocessing": "SoftCom_02_Preprocessing.ipynb",
    "03_benchmark": "SoftCom_03_Benchmarking.ipynb",
    "04_xai": "SoftCom_04_Explainable_AI.ipynb",
    "05_figures": "SoftCom_05_Figures.ipynb",
    "06_export": "SoftCom_06_Export.ipynb",
}

# ----------------------------------------------------------------------------- Drive
LAUNCH_DIR: Path = Path.cwd()
try:
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive")
        except Exception as exc:  # pragma: no cover - interactive auth
            raise RuntimeError(
                "Google Drive mount failed. Re-run this cell and accept the Drive "
                f"authorisation prompt. Original error: {exc}"
            ) from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive") / CONFIG["DRIVE_PROJECT_DIR"]
else:
    PROJECT_ROOT = Path(os.environ.get("SOFTCOM_ROOT", LAUNCH_DIR / CONFIG["DRIVE_PROJECT_DIR"])).resolve()

DIRS: Dict[str, Path] = {
    name: PROJECT_ROOT / sub
    for name, sub in {
        "raw": "data/raw",
        "processed": "data/processed",
        "models": "models",
        "predictions": "predictions",
        "tables": "tables",
        "figures": "figures",
        "artifacts": "artifacts",
        "logs": "logs",
    }.items()
}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)  # so ./figures/ and ./tables/ resolve inside Google Drive
MANIFEST_PATH: Path = DIRS["artifacts"] / "manifest.json"


# ----------------------------------------------------------------------------- helpers
def set_global_seed(seed: int = SEED) -> None:
    """Seed python, numpy and (if installed) torch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch  # type: ignore

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass


def log(step: str, message: str) -> None:
    """Print a message and append it to logs/<step>.log on Drive."""
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    print(message)
    with open(DIRS["logs"] / f"{step}.log", "a", encoding="utf-8") as fh:
        fh.write(f"[{stamp}] {message}\n")


def sha256_file(path: Path) -> str:
    """Return the SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def rel(path: Path) -> str:
    """Path relative to the project root (as stored in the manifest)."""
    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def _atomic_write_json(obj: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


def save_json(obj: Any, path: Path) -> Path:
    """Write JSON atomically (numpy/pandas values are stringified if needed)."""
    _atomic_write_json(obj, path)
    return path


def load_manifest() -> Dict[str, Any]:
    """Load the cross-notebook run manifest (empty skeleton if absent)."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, encoding="utf-8") as fh:
            return json.load(fh)
    return {"project": "SoftCom myopia benchmark", "steps": {}}


def register_outputs(step: str, paths: Iterable[Path], extra: Optional[Dict[str, Any]] = None) -> None:
    """Record a finished notebook step and the SHA-256 of every file it wrote."""
    manifest = load_manifest()
    files = {}
    for p in paths:
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"[{step}] expected output was not written: {p}")
        files[rel(p)] = sha256_file(p)
    manifest["steps"][step] = {
        "notebook": NOTEBOOKS[step],
        "status": "ok",
        "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "fast_mode": CONFIG["FAST_MODE"],
        "outputs": files,
        "extra": extra or {},
    }
    _atomic_write_json(manifest, MANIFEST_PATH)
    print(f"Manifest updated: step '{step}' -> {len(files)} files registered.")


def require_upstream(step: str, needed: Sequence[str]) -> None:
    """Fail loudly unless `step` ran successfully and its files are unchanged.

    `needed` are project-relative paths that this notebook will read.
    """
    manifest = load_manifest()
    info = manifest["steps"].get(step)
    if info is None or info.get("status") != "ok":
        raise RuntimeError(
            f"Upstream step '{step}' has not completed. Open and 'Run all' "
            f"{NOTEBOOKS[step]} first (results folder: {PROJECT_ROOT})."
        )
    for r in needed:
        p = PROJECT_ROOT / r
        if not p.exists():
            raise FileNotFoundError(f"Missing upstream file {r}; re-run {NOTEBOOKS[step]}.")
        expected = info["outputs"].get(r)
        if expected is None:
            raise RuntimeError(f"{r} is not registered by {NOTEBOOKS[step]}; re-run it.")
        if sha256_file(p) != expected:
            raise RuntimeError(
                f"{r} changed after {NOTEBOOKS[step]} finished (hash drift). "
                f"Re-run {NOTEBOOKS[step]} and the notebooks after it."
            )
    if info.get("fast_mode") != CONFIG["FAST_MODE"]:
        print(f"WARNING: {step} ran with FAST_MODE={info.get('fast_mode')}, this notebook uses "
              f"FAST_MODE={CONFIG['FAST_MODE']}.")
    print(f"Upstream OK: {step} ({info['finished_at']}), {len(needed)} files verified.")


sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "legend.fontsize": 10, "savefig.bbox": "tight"})


def save_figure(fig: plt.Figure, name: str, footnote: bool = True) -> List[Path]:
    """Save `fig` as 300-DPI PNG and SVG in ./figures/ (on Drive) and close it."""
    if footnote:  # place below everything already drawn (rotated tick labels included)
        bottom = fig.get_tightbbox(fig.canvas.get_renderer()).y0 / fig.get_figheight()
        fig.text(0.995, min(bottom, 0.0) - 0.01, CONFIG["COHORT_LABEL"], ha="right", va="top", fontsize=8, color="0.45")
    out = []
    try:
        for ext in ("png", "svg"):
            p = DIRS["figures"] / f"{name}.{ext}"
            fig.savefig(p, dpi=300, bbox_inches="tight")
            out.append(p)
    except OSError as exc:
        raise RuntimeError(f"Could not write figure {name} to {DIRS['figures']}: {exc}") from exc
    finally:
        plt.close(fig)
    print(f"Saved figure: figures/{name}.png + .svg")
    return out


def show_saved(name: str) -> None:
    """Display a saved PNG inline (Colab/Jupyter only)."""
    try:
        from IPython.display import Image, display  # type: ignore

        if "ipykernel" in sys.modules:
            display(Image(filename=str(DIRS["figures"] / f"{name}.png"), width=720))
    except ImportError:
        pass


set_global_seed(SEED)
print(f"Colab: {IN_COLAB} | results folder: {PROJECT_ROOT} | FAST_MODE={CONFIG['FAST_MODE']}")

## 3.1 Model library and registry
Adding a model = one decorated factory returning `(pipeline, param_grid)`; the benchmark loop needs no edits.

**Architecture rationale (extended models).** The FT-Transformer tokenises every feature (a learned
vector per numeric feature, an embedding per categorical code) and lets self-attention model
pairwise interactions such as axial length x parental myopia, which tree ensembles approximate with
many splits and linear models ignore. With only 12 features and ~1,000 training rows it is at real risk
of overfitting, so it uses 2 small layers, dropout, weight decay and early stopping on inner-fold AUC.
The ablation `FT-Transformer-noAttn` (enabled by `CONFIG['ABLATION']`) keeps the tokenizer and head but
removes attention, isolating what attention adds. The stacking ensemble combines LR/RF/XGB/SVM
out-of-fold probabilities with a logistic meta-learner to test whether heterogeneous inductive biases
are complementary. Per-fold training curves are saved to `figures/training_curves_*.png`.

In [ ]:
"""Model library: pipeline builder, MODEL_REGISTRY, PyTorch tabular models, metrics, DeLong.

Identical in notebooks 03 and 04 so fitted models saved by 03 can be unpickled by 04.
"""
import copy
from dataclasses import dataclass, field
from typing import Callable, Tuple

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

OPTIONAL_LIBS: Dict[str, bool] = {}
try:
    from xgboost import XGBClassifier  # type: ignore
    OPTIONAL_LIBS["xgboost"] = True
except ImportError:
    OPTIONAL_LIBS["xgboost"] = False
try:
    from lightgbm import LGBMClassifier  # type: ignore
    OPTIONAL_LIBS["lightgbm"] = True
except ImportError:
    OPTIONAL_LIBS["lightgbm"] = False
try:
    from imblearn.over_sampling import SMOTE  # type: ignore
    from imblearn.pipeline import Pipeline as ImbPipeline  # type: ignore
    OPTIONAL_LIBS["imblearn"] = True
except ImportError:
    OPTIONAL_LIBS["imblearn"] = False
try:
    import torch  # type: ignore
    from torch import nn  # type: ignore
    OPTIONAL_LIBS["torch"] = True
except ImportError:
    OPTIONAL_LIBS["torch"] = False
print("Optional libraries:", OPTIONAL_LIBS)
if CONFIG["USE_SMOTE"] and not OPTIONAL_LIBS["imblearn"]:
    raise ImportError("CONFIG['USE_SMOTE']=True needs imbalanced-learn: run the install cell.")

SKLEARN_VERSION: Tuple[int, int] = tuple(int(v) for v in sklearn.__version__.split(".")[:2])  # type: ignore


def l1_logistic(C: float = 0.5) -> LogisticRegression:
    """L1-penalised logistic regression, compatible with sklearn <1.8 and >=1.8."""
    kw: Dict[str, Any] = dict(C=C, solver="liblinear", class_weight="balanced", random_state=SEED, max_iter=2000)
    if SKLEARN_VERSION >= (1, 8):
        kw["l1_ratio"] = 1.0
    else:
        kw["penalty"] = "l1"
    return LogisticRegression(**kw)


# ----------------------------------------------------------------------------- pipeline
@dataclass
class FactoryContext:
    """Everything a model factory needs to build its pipeline."""

    numeric: List[str]
    categorical: List[str]
    cat_cardinalities: List[int]
    pos_weight: float
    seed: int = SEED


def build_preprocessor(numeric: List[str], categorical: List[str]) -> ColumnTransformer:
    """Impute+scale numeric, impute categorical codes. Output order: numeric, categorical."""
    num = SkPipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat = SimpleImputer(strategy="most_frequent")
    return ColumnTransformer([("num", num, numeric), ("cat", cat, categorical)],
                             verbose_feature_names_out=False)


def make_pipeline(model: Any, ctx: FactoryContext, use_selector: bool = True) -> Any:
    """imputer -> scaler -> [SMOTE] -> LASSO selector -> model, all fitted inside CV folds."""
    steps: List[Tuple[str, Any]] = [("prep", build_preprocessor(ctx.numeric, ctx.categorical))]
    if CONFIG["USE_SMOTE"]:
        steps.append(("smote", SMOTE(random_state=ctx.seed)))
    if use_selector and CONFIG["USE_LASSO_SELECTION"]:
        steps.append(("select", SelectFromModel(l1_logistic(0.5), threshold=1e-5)))
    steps.append(("model", model))
    return ImbPipeline(steps) if CONFIG["USE_SMOTE"] else SkPipeline(steps)


# ----------------------------------------------------------------------------- torch models
if OPTIONAL_LIBS["torch"]:

    class _MLPNet(nn.Module):
        """MLP with entity embeddings for categorical codes."""

        def __init__(self, n_num: int, cards: Sequence[int], hidden: int, dropout: float) -> None:
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c, min(4, (c + 1) // 2 + 1)) for c in cards])
            d_in = n_num + sum(e.embedding_dim for e in self.embs)
            self.body = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))

        def forward(self, x_num: "torch.Tensor", x_cat: "torch.Tensor") -> "torch.Tensor":
            parts = [x_num] + [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
            return self.body(torch.cat(parts, dim=1)).squeeze(1)

    class _FTTNet(nn.Module):
        """Minimal FT-Transformer: feature tokenizer + [CLS] + TransformerEncoder."""

        def __init__(self, n_num: int, cards: Sequence[int], d: int, n_heads: int, n_layers: int,
                     dropout: float, use_attention: bool) -> None:
            super().__init__()
            self.num_w = nn.Parameter(torch.randn(n_num, d) * 0.1)
            self.num_b = nn.Parameter(torch.zeros(n_num, d))
            self.cat_embs = nn.ModuleList([nn.Embedding(c, d) for c in cards])
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            self.use_attention = use_attention
            if use_attention:
                layer = nn.TransformerEncoderLayer(d_model=d, nhead=n_heads, dim_feedforward=2 * d,
                                                   dropout=dropout, batch_first=True, norm_first=True)
                self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
            else:  # ablation: same tokenizer and head, no self-attention
                self.encoder = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout))
            self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, 1))

        def forward(self, x_num: "torch.Tensor", x_cat: "torch.Tensor") -> "torch.Tensor":
            tokens = [x_num.unsqueeze(-1) * self.num_w + self.num_b]
            if len(self.cat_embs):
                tokens.append(torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embs)], dim=1))
            t = torch.cat(tokens, dim=1)
            if self.use_attention:
                t = torch.cat([self.cls.expand(t.shape[0], -1, -1), t], dim=1)
                return self.head(self.encoder(t)[:, 0]).squeeze(1)
            return self.head(self.encoder(t).mean(dim=1)).squeeze(1)


class TorchTabularClassifier(ClassifierMixin, BaseEstimator):
    """sklearn-compatible PyTorch classifier (arch='mlp' or 'ftt').

    Expects the last `len(cat_cardinalities)` input columns to be integer category codes
    (the preprocessor output order). Early-stops on the AUC of an inner stratified
    validation split carved out of the training fold only.
    """

    def __init__(self, arch: str = "mlp", cat_cardinalities: Tuple[int, ...] = (), hidden: int = 16,
                 d_token: int = 16, n_heads: int = 2, n_layers: int = 2, dropout: float = 0.3,
                 lr: float = 1e-3, weight_decay: float = 0.01, batch_size: int = 64, max_epochs: int = 200,
                 patience: int = 20, val_fraction: float = 0.15, use_attention: bool = True,
                 balanced: bool = True, random_state: int = SEED) -> None:
        self.arch = arch
        self.cat_cardinalities = cat_cardinalities
        self.hidden = hidden
        self.d_token = d_token
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.val_fraction = val_fraction
        self.use_attention = use_attention
        self.balanced = balanced
        self.random_state = random_state

    def _split(self, X: np.ndarray) -> Tuple["torch.Tensor", "torch.Tensor"]:
        n_cat = len(self.cat_cardinalities)
        n_num = X.shape[1] - n_cat
        x_num = torch.tensor(X[:, :n_num], dtype=torch.float32)
        cat = np.rint(X[:, n_num:]).astype(np.int64)
        if n_cat:
            cat = np.clip(cat, 0, np.asarray(self.cat_cardinalities) - 1)
        return x_num, torch.tensor(cat, dtype=torch.long)

    def _build(self, n_num: int) -> "nn.Module":
        if self.arch == "mlp":
            return _MLPNet(n_num, self.cat_cardinalities, self.hidden, self.dropout)
        if self.arch == "ftt":
            return _FTTNet(n_num, self.cat_cardinalities, self.d_token, self.n_heads, self.n_layers,
                           self.dropout, self.use_attention)
        raise ValueError(f"Unknown arch {self.arch!r}")

    def fit(self, X: Any, y: Any) -> "TorchTabularClassifier":
        """Train with AdamW (L2 via weight_decay) and AUC-based early stopping."""
        if not OPTIONAL_LIBS["torch"]:
            raise ImportError("PyTorch is required for TorchTabularClassifier.")
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y).astype(int)
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = X.shape[1]
        torch.manual_seed(self.random_state)
        Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=self.val_fraction, stratify=y,
                                              random_state=self.random_state)
        xn_tr, xc_tr = self._split(Xtr)
        xn_va, xc_va = self._split(Xva)
        yt = torch.tensor(ytr, dtype=torch.float32)
        net = self._build(xn_tr.shape[1])
        pos_w = (len(ytr) - ytr.sum()) / max(ytr.sum(), 1) if self.balanced else 1.0
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_w)))
        opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        gen = torch.Generator().manual_seed(self.random_state)
        best_auc, best_state, wait = -np.inf, None, 0
        self.history_: Dict[str, List[float]] = {"train_loss": [], "val_loss": [], "val_auc": []}
        for _epoch in range(self.max_epochs):
            net.train()
            perm = torch.randperm(len(yt), generator=gen)
            losses = []
            for i in range(0, len(perm), self.batch_size):
                idx = perm[i:i + self.batch_size]
                opt.zero_grad()
                loss = loss_fn(net(xn_tr[idx], xc_tr[idx]), yt[idx])
                loss.backward()
                opt.step()
                losses.append(loss.item())
            net.eval()
            with torch.no_grad():
                logit_va = net(xn_va, xc_va)
                val_loss = loss_fn(logit_va, torch.tensor(yva, dtype=torch.float32)).item()
                val_auc = roc_auc_score(yva, torch.sigmoid(logit_va).numpy())
            self.history_["train_loss"].append(float(np.mean(losses)))
            self.history_["val_loss"].append(val_loss)
            self.history_["val_auc"].append(float(val_auc))
            if val_auc > best_auc + 1e-4:
                best_auc, best_state, wait = val_auc, copy.deepcopy(net.state_dict()), 0
                self.best_epoch_ = _epoch
            else:
                wait += 1
                if wait >= self.patience:
                    break
        net.load_state_dict(best_state)
        net.eval()
        self.net_ = net
        self.n_params_ = int(sum(p.numel() for p in net.parameters()))
        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        """Return [P(y=0), P(y=1)] per row."""
        xn, xc = self._split(np.asarray(X, dtype=np.float64))
        with torch.no_grad():
            p = torch.sigmoid(self.net_(xn, xc)).numpy().astype(np.float64)
        return np.column_stack([1.0 - p, p])

    def predict(self, X: Any) -> np.ndarray:
        """Class labels at the 0.5 threshold."""
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ----------------------------------------------------------------------------- registry
ModelSpec = Tuple[Any, Dict[str, List[Any]]]
MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {}


def register_model(name: str, group: str, kind: str) -> Callable:
    """Decorator: add a factory(ctx) -> (pipeline, param_grid) to MODEL_REGISTRY.

    group: 'baseline' | 'extended' | 'ablation'; kind: 'tree' | 'kernel' (SHAP explainer family).
    A factory returning None is skipped with a logged reason (missing optional library).
    """

    def deco(fn: Callable[[FactoryContext], Optional[ModelSpec]]) -> Callable:
        MODEL_REGISTRY[name] = {"factory": fn, "group": group, "kind": kind}
        return fn

    return deco


# Baselines (frozen names and search spaces - Mu et al. 2024 replica)
@register_model("LR", "baseline", "kernel")
def _lr(ctx: FactoryContext) -> ModelSpec:
    m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=ctx.seed)
    return make_pipeline(m, ctx), {"model__C": [1.0]}


@register_model("DT", "baseline", "tree")
def _dt(ctx: FactoryContext) -> ModelSpec:
    m = DecisionTreeClassifier(class_weight="balanced", random_state=ctx.seed)
    return make_pipeline(m, ctx), {"model__max_depth": [2, 3, 4]}


@register_model("RF", "baseline", "tree")
def _rf(ctx: FactoryContext) -> ModelSpec:
    m = RandomForestClassifier(class_weight="balanced", random_state=ctx.seed, n_jobs=-1)
    return make_pipeline(m, ctx), {"model__n_estimators": [100, 300]}


def _xgb_estimator(ctx: FactoryContext, **kw: Any) -> Any:
    return XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                         eval_metric="logloss", scale_pos_weight=ctx.pos_weight, random_state=ctx.seed,
                         n_jobs=-1, **kw)


@register_model("XGBoost", "baseline", "tree")
def _xgb(ctx: FactoryContext) -> Optional[ModelSpec]:
    if not OPTIONAL_LIBS["xgboost"]:
        return None
    grid = {"model__learning_rate": [0.05, 0.1], "model__max_depth": [2, 3, 4], "model__min_child_weight": [5]}
    return make_pipeline(_xgb_estimator(ctx), ctx), grid


@register_model("SVM", "baseline", "kernel")
def _svm(ctx: FactoryContext) -> ModelSpec:
    m = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=ctx.seed)
    return make_pipeline(m, ctx), {"model__C": [0.1, 1.0, 10.0]}


# Extended models (our contribution)
@register_model("LightGBM", "extended", "tree")
def _lgbm(ctx: FactoryContext) -> Optional[ModelSpec]:
    if not OPTIONAL_LIBS["lightgbm"]:
        return None
    m = LGBMClassifier(n_estimators=300, max_depth=3, min_child_samples=20, subsample=0.8, subsample_freq=1,
                       colsample_bytree=0.8, class_weight="balanced", random_state=ctx.seed, n_jobs=-1,
                       verbose=-1)
    return make_pipeline(m, ctx), {"model__num_leaves": [7, 15], "model__learning_rate": [0.05, 0.1]}


@register_model("Stacking", "extended", "kernel")
def _stack(ctx: FactoryContext) -> ModelSpec:
    base = [("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=ctx.seed)),
            ("rf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=ctx.seed,
                                          n_jobs=-1)),
            ("svm", SVC(kernel="rbf", C=1.0, probability=True, class_weight="balanced", random_state=ctx.seed))]
    if OPTIONAL_LIBS["xgboost"]:
        base.insert(2, ("xgb", _xgb_estimator(ctx, learning_rate=0.05, max_depth=3, min_child_weight=5)))
    m = StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=1000),
                           cv=StratifiedKFold(5, shuffle=True, random_state=ctx.seed),
                           stack_method="predict_proba")
    return make_pipeline(m, ctx), {"model__final_estimator__C": [0.1, 1.0]}


def _torch_kw(ctx: FactoryContext) -> Dict[str, Any]:
    return dict(cat_cardinalities=tuple(ctx.cat_cardinalities), batch_size=CONFIG["DEEP_BATCH_SIZE"],
                max_epochs=CONFIG["DEEP_MAX_EPOCHS"], patience=CONFIG["DEEP_PATIENCE"], random_state=ctx.seed)


@register_model("MLP", "extended", "kernel")
def _mlp(ctx: FactoryContext) -> ModelSpec:
    if OPTIONAL_LIBS["torch"]:
        m = TorchTabularClassifier(arch="mlp", hidden=16, dropout=0.3, weight_decay=0.01, **_torch_kw(ctx))
        return make_pipeline(m, ctx, use_selector=False), {"model__lr": [1e-3, 3e-3]}
    print("PyTorch missing -> MLP falls back to sklearn MLPClassifier (no dropout/embeddings).")
    m = MLPClassifier(hidden_layer_sizes=(16,), alpha=0.01, early_stopping=True, max_iter=500,
                      random_state=ctx.seed)
    return make_pipeline(m, ctx, use_selector=False), {"model__learning_rate_init": [1e-3, 3e-3]}


@register_model("FT-Transformer", "extended", "kernel")
def _ftt(ctx: FactoryContext) -> Optional[ModelSpec]:
    if not OPTIONAL_LIBS["torch"]:
        return None
    m = TorchTabularClassifier(arch="ftt", n_heads=2, n_layers=2, dropout=0.1, lr=1e-3, weight_decay=1e-4,
                               **_torch_kw(ctx))
    return make_pipeline(m, ctx, use_selector=False), {"model__d_token": [16, 32]}


@register_model("FT-Transformer-noAttn", "ablation", "kernel")
def _ftt_ablation(ctx: FactoryContext) -> Optional[ModelSpec]:
    if not (OPTIONAL_LIBS["torch"] and CONFIG["ABLATION"]):
        return None
    m = TorchTabularClassifier(arch="ftt", use_attention=False, dropout=0.1, lr=1e-3, weight_decay=1e-4,
                               **_torch_kw(ctx))
    return make_pipeline(m, ctx, use_selector=False), {"model__d_token": [16, 32]}


# ----------------------------------------------------------------------------- metrics
def classification_metrics(y: np.ndarray, p: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    """Accuracy, precision, recall, F1 at `threshold` plus ROC-AUC."""
    yhat = (p >= threshold).astype(int)
    return {"accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),
            "recall": recall_score(y, yhat, zero_division=0), "f1": f1_score(y, yhat, zero_division=0),
            "roc_auc": roc_auc_score(y, p)}


def bootstrap_auc_ci(y: np.ndarray, p: np.ndarray, n_boot: int = 2000, seed: int = SEED,
                     alpha: float = 0.05) -> Tuple[float, float]:
    """Percentile bootstrap CI for ROC-AUC (resamples lacking a class are skipped)."""
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return float(np.quantile(aucs, alpha / 2)), float(np.quantile(aucs, 1 - alpha / 2))


def _midrank(x: np.ndarray) -> np.ndarray:
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = t
    return out


def delong_test(y: np.ndarray, p1: np.ndarray, p2: np.ndarray) -> Tuple[float, float, float, float]:
    """Paired DeLong test (Sun & Xu 2014 fast algorithm). Returns (auc1, auc2, z, two-sided p)."""
    from scipy.stats import norm

    y = np.asarray(y).astype(int)
    order = np.argsort(-y, kind="mergesort")  # positives first
    preds = np.vstack([np.asarray(p1)[order], np.asarray(p2)[order]])
    m = int(y.sum())
    n = len(y) - m
    tx = np.array([_midrank(r[:m]) for r in preds])
    ty = np.array([_midrank(r[m:]) for r in preds])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * norm.sf(abs(z)))


def holm_adjust(pvals: Sequence[float]) -> np.ndarray:
    """Holm-Bonferroni step-down adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[idx]))
        adj[idx] = running
    return adj


print("Registered models:", {k: v["group"] for k, v in MODEL_REGISTRY.items()})

## 3.2 Load the split from notebook 02

In [ ]:
STEP = "03_benchmark"
require_upstream("02_preprocessing", ["data/processed/cohort_features.csv", "data/processed/train.csv",
                                      "data/processed/test.csv", "artifacts/feature_spec.json"])
with open(DIRS["artifacts"] / "feature_spec.json", encoding="utf-8") as fh:
    FSPEC = json.load(fh)
TARGET = FSPEC["target"]
FEATURES = FSPEC["numeric"] + FSPEC["categorical"]
full_df = pd.read_csv(DIRS["processed"] / "cohort_features.csv")
train_df = pd.read_csv(DIRS["processed"] / "train.csv")
test_df = pd.read_csv(DIRS["processed"] / "test.csv")
X_all, y_all = full_df[FEATURES], full_df[TARGET].values
X_tr, y_tr = train_df[FEATURES], train_df[TARGET].values
X_te, y_te = test_df[FEATURES], test_df[TARGET].values
ctx = FactoryContext(numeric=FSPEC["numeric"], categorical=FSPEC["categorical"],
                     cat_cardinalities=FSPEC["cat_cardinalities"], pos_weight=float((y_tr == 0).sum() / (y_tr == 1).sum()))
print(f"all={len(X_all)} train={len(X_tr)} test={len(X_te)} features={len(FEATURES)} pos_weight={ctx.pos_weight:.2f}")

## 3.3 Hyperparameter tuning (inner 5-fold CV on the 70% training split)

In [ ]:
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

tuned: Dict[str, Any] = {}
meta: Dict[str, Dict[str, Any]] = {}
skipped: Dict[str, str] = {}
inner_cv = StratifiedKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=SEED)
for name, entry in MODEL_REGISTRY.items():
    spec_ = entry["factory"](ctx)
    if spec_ is None:
        skipped[name] = "optional library missing or disabled by CONFIG"
        log(STEP, f"SKIP {name}: {skipped[name]}")
        continue
    pipe, grid = spec_
    set_global_seed(SEED)
    t0 = time.time()
    try:
        gs = GridSearchCV(pipe, grid, scoring="roc_auc", cv=inner_cv, n_jobs=1, refit=True, error_score="raise")
        gs.fit(X_tr, y_tr)
    except Exception as exc:
        skipped[name] = f"tuning failed: {type(exc).__name__}: {exc}"
        log(STEP, f"FAIL {name}: {skipped[name]} -> check the factory/param grid for this model.")
        continue
    tuned[name] = gs.best_estimator_
    n_cfg = int(np.prod([len(v) for v in grid.values()]))
    meta[name] = {"group": entry["group"], "kind": entry["kind"], "best_params": gs.best_params_,
                  "inner_cv_auc": float(gs.best_score_), "n_grid_configs": n_cfg,
                  "tuning_seconds": round(time.time() - t0, 1)}
    log(STEP, f"{name:<22} inner AUC={gs.best_score_:.3f} params={gs.best_params_} ({meta[name]['tuning_seconds']}s)")

missing_baselines = [m for m in ["LR", "DT", "RF", "XGBoost", "SVM"] if m not in tuned]
if missing_baselines:
    raise RuntimeError(f"Mandatory baselines missing: {missing_baselines}. Run the install cell and retry. {skipped}")

## 3.4 Headline evaluation: RepeatedStratifiedKFold (5 x 10) on the full cohort
Hyper-parameters come from 3.3 (chosen on the training split only). Every pipeline is refitted from
scratch in every fold, so imputation, scaling, LASSO selection and SMOTE never see the fold's test rows.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=CONFIG["N_SPLITS"], n_repeats=CONFIG["N_REPEATS"], random_state=SEED)
splits = list(rskf.split(X_all, y_all))
fold_hash = hashlib.sha256("|".join(",".join(map(str, te)) for _, te in splits).encode()).hexdigest()
print(f"{len(splits)} folds | fold-index hash {fold_hash[:16]}")

fold_rows: List[Dict[str, Any]] = []
oof = pd.DataFrame({"row_id": full_df["row_id"], "y": y_all})
curves: Dict[str, List[Dict[str, List[float]]]] = {}
for name, est in list(tuned.items()):
    t0 = time.time()
    oof[name] = np.nan
    try:
        for k, (tr, te) in enumerate(splits):
            set_global_seed(SEED)
            model = clone(est).fit(X_all.iloc[tr], y_all[tr])
            p = model.predict_proba(X_all.iloc[te])[:, 1]
            fold_rows.append({"model": name, "repeat": k // CONFIG["N_SPLITS"], "fold": k % CONFIG["N_SPLITS"],
                              **classification_metrics(y_all[te], p)})
            if k < CONFIG["N_SPLITS"]:  # repeat 0 -> pooled out-of-fold predictions
                oof.loc[oof.index[te], name] = p
                hist = getattr(model.named_steps["model"], "history_", None)
                if hist is not None:
                    curves.setdefault(name, []).append(hist)
    except Exception as exc:
        log(STEP, f"FAIL CV {name}: {type(exc).__name__}: {exc} - model dropped from comparison")
        skipped[name] = f"CV failed: {exc}"
        tuned.pop(name, None)
        oof = oof.drop(columns=[name])
        fold_rows = [r for r in fold_rows if r["model"] != name]
        continue
    meta[name]["cv_seconds"] = round(time.time() - t0, 1)
    m = pd.DataFrame([r for r in fold_rows if r["model"] == name])
    log(STEP, f"{name:<22} CV AUC {m.roc_auc.mean():.3f} +- {m.roc_auc.std():.3f} ({meta[name]['cv_seconds']}s)")

fold_df = pd.DataFrame(fold_rows)
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_summary = fold_df.groupby("model", sort=False)[METRICS].agg(["mean", "std"])
cv_table = pd.DataFrame({m: cv_summary[(m, "mean")].map("{:.3f}".format) + " +- " + cv_summary[(m, "std")].map("{:.3f}".format)
                         for m in METRICS})
print(cv_table.to_string())

## 3.5 Paper replica: fixed 70/30 holdout with bootstrap 95% CIs

In [ ]:
hold_rows, pred_te, pred_tr = [], pd.DataFrame({"row_id": test_df.row_id, "y": y_te}), pd.DataFrame({"row_id": train_df.row_id, "y": y_tr})
for name, est in tuned.items():
    pred_tr[name] = est.predict_proba(X_tr)[:, 1]
    pred_te[name] = est.predict_proba(X_te)[:, 1]
    met = classification_metrics(y_te, pred_te[name].values)
    lo, hi = bootstrap_auc_ci(y_te, pred_te[name].values, CONFIG["N_BOOTSTRAP"])
    tlo, thi = bootstrap_auc_ci(y_tr, pred_tr[name].values, CONFIG["N_BOOTSTRAP"])
    olo, ohi = bootstrap_auc_ci(y_all, oof[name].values, CONFIG["N_BOOTSTRAP"])
    hold_rows.append({"model": name, "group": meta[name]["group"], **met, "auc_ci_low": lo, "auc_ci_high": hi,
                      "train_auc": roc_auc_score(y_tr, pred_tr[name]), "train_auc_ci_low": tlo, "train_auc_ci_high": thi,
                      "oof_auc": roc_auc_score(y_all, oof[name]), "oof_auc_ci_low": olo, "oof_auc_ci_high": ohi})
holdout = pd.DataFrame(hold_rows).set_index("model")
print(holdout[["accuracy", "precision", "recall", "f1", "roc_auc", "auc_ci_low", "auc_ci_high"]].round(3).to_string())

## 3.6 DeLong pairwise AUC comparison on the holdout (Holm-corrected)

In [ ]:
from itertools import combinations

dl_rows = []
for a, b in combinations(tuned, 2):
    auc_a, auc_b, z, p = delong_test(y_te, pred_te[a].values, pred_te[b].values)
    dl_rows.append({"model_a": a, "model_b": b, "auc_a": auc_a, "auc_b": auc_b, "delta_auc": auc_a - auc_b, "z": z, "p_raw": p})
delong_df = pd.DataFrame(dl_rows)
delong_df["p_holm"] = holm_adjust(delong_df["p_raw"].values)
delong_df["significant_0.05"] = delong_df["p_holm"] < 0.05
assert np.allclose(delong_df["auc_a"], [roc_auc_score(y_te, pred_te[a]) for a in delong_df.model_a]), "DeLong AUC mismatch"
print(delong_df.round(4).to_string(index=False))

## 3.7 Table 2 - model performance (best row = highest repeated-CV AUC, shown bold)

In [ ]:
table2 = pd.DataFrame({
    "Group": holdout["group"],
    "Accuracy": holdout["accuracy"].round(3), "Precision": holdout["precision"].round(3),
    "Recall": holdout["recall"].round(3), "F1": holdout["f1"].round(3), "AUC": holdout["roc_auc"].round(3),
    "AUC 95% CI": holdout.apply(lambda r: f"{r.auc_ci_low:.3f}-{r.auc_ci_high:.3f}", axis=1),
    "CV AUC (mean +- SD)": cv_table["roc_auc"], "CV F1 (mean +- SD)": cv_table["f1"],
    "CV Accuracy (mean +- SD)": cv_table["accuracy"],
})
best_model = cv_summary[("roc_auc", "mean")].idxmax()
table2["Best"] = np.where(table2.index == best_model, "*", "")
table2.index.name = "Model"


def to_markdown_bold(df_: pd.DataFrame, bold_row: str) -> str:
    """Render a DataFrame as a GitHub markdown table with one bold row (no tabulate dependency)."""
    cols = [df_.index.name] + list(df_.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in df_.iterrows():
        cells = [str(idx)] + [str(v) for v in row.values]
        if idx == bold_row:
            cells = [f"**{c}**" if c else c for c in cells]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


print(table2.to_string())
print(f"\nBest model by repeated-CV AUC: {best_model}  ({CONFIG['COHORT_LABEL']})")

## 3.8 Training curves for deep models (per fold, CV repeat 0)

In [ ]:
curve_paths: List[Path] = []
for name, hists in curves.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for i, h in enumerate(hists):
        axes[0].plot(h["train_loss"], lw=1.2, label=f"fold {i + 1}")
        axes[1].plot(h["val_auc"], lw=1.2, label=f"fold {i + 1}")
    axes[0].set(title=f"{name}: training loss", xlabel="epoch", ylabel="BCE loss")
    axes[1].set(title=f"{name}: inner-validation AUC", xlabel="epoch", ylabel="AUC", ylim=(0.4, 1.0))
    axes[1].legend(fontsize=9)
    curve_paths += save_figure(fig, f"training_curves_{name.replace(' ', '_')}")
if not curves:
    print("No deep-model training curves (PyTorch models not trained).")

## 3.9 Save models, predictions and tables to Drive

In [ ]:
import joblib

paths: List[Path] = []
for name, est in tuned.items():
    p = DIRS["models"] / f"{name}.joblib"
    try:
        joblib.dump(est, p)
    except Exception as exc:
        raise RuntimeError(f"Could not save model {name} to Drive: {exc}") from exc
    paths.append(p)
    if hasattr(est.named_steps["model"], "n_params_"):
        meta[name]["n_parameters"] = est.named_steps["model"].n_params_

out = {
    "table2": DIRS["tables"] / "table2_model_performance.csv",
    "table2_md": DIRS["tables"] / "table2_model_performance.md",
    "table2_html": DIRS["tables"] / "table2_model_performance.html",
    "cv_summary": DIRS["tables"] / "table2b_repeated_cv_summary.csv",
    "delong": DIRS["tables"] / "table3_delong_pairwise.csv",
    "holdout_full": DIRS["tables"] / "table2c_holdout_detail.csv",
    "folds": DIRS["predictions"] / "cv_fold_metrics.csv",
    "oof": DIRS["predictions"] / "oof_predictions_repeat0.csv",
    "pred_te": DIRS["predictions"] / "holdout_test_predictions.csv",
    "pred_tr": DIRS["predictions"] / "holdout_train_predictions.csv",
    "folds_json": DIRS["artifacts"] / "fold_indices.json",
    "meta": DIRS["artifacts"] / "benchmark_meta.json",
}
table2.to_csv(out["table2"])
out["table2_md"].write_text(to_markdown_bold(table2, best_model) + f"\n\n_{CONFIG['COHORT_LABEL']}_\n", encoding="utf-8")
try:
    html = table2.style.apply(lambda r: ["font-weight: bold" if r.name == best_model else "" for _ in r], axis=1).to_html()
except ImportError:  # Styler needs jinja2
    html = table2.to_html().replace(f"<th>{best_model}</th>", f"<th><b>{best_model}</b></th>")
out["table2_html"].write_text(html, encoding="utf-8")
cv_summary.to_csv(out["cv_summary"])
delong_df.to_csv(out["delong"], index=False)
holdout.to_csv(out["holdout_full"])
fold_df.to_csv(out["folds"], index=False)
oof.to_csv(out["oof"], index=False)
pred_te.to_csv(out["pred_te"], index=False)
pred_tr.to_csv(out["pred_tr"], index=False)
save_json({"hash": fold_hash, "n_splits": CONFIG["N_SPLITS"], "n_repeats": CONFIG["N_REPEATS"],
           "test_indices": [te.tolist() for _, te in splits]}, out["folds_json"])
save_json({"models": meta, "skipped": skipped, "best_model": best_model, "fold_hash": fold_hash,
           "optional_libs": OPTIONAL_LIBS, "config": CONFIG,
           "note": "Hyper-parameters tuned on the 70% split; repeated CV on the full cohort with those settings."},
          out["meta"])
register_outputs(STEP, paths + list(out.values()) + curve_paths, {"best_model": best_model, "n_models": len(tuned)})

In [ ]:
print("=" * 70)
print(f"03 DONE | models={len(tuned)} ({', '.join(tuned)}) | skipped={list(skipped) or 'none'}")
print(f"best (CV AUC) = {best_model}: {cv_table.loc[best_model, 'roc_auc']} | holdout AUC "
      f"{holdout.loc[best_model, 'roc_auc']:.3f} ({holdout.loc[best_model, 'auc_ci_low']:.3f}-{holdout.loc[best_model, 'auc_ci_high']:.3f})")
print(f"NEXT -> open {NOTEBOOKS['04_xai']} and Run all")